# Run inference for Qwen3TTS

This notebook demonstrates how to:
1. Initialize the Qwen3TTS model with dummy weights
2. Run inference with random embeddings to verify the model works


In [ ]:
import os
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"


In [ ]:
import torch
from pathlib import Path
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

print("Imports successful")


In [ ]:
# Load vLLM engine with dummy model
type_str = "bfloat16"
torch_type = getattr(torch, type_str)

max_len = 256
config_path = Path("dummy_qwen3_tts_model")
engine_args = AsyncEngineArgs(
    model=str(config_path.absolute()),
    dtype=type_str,
    max_model_len=max_len,
    max_num_batched_tokens=max_len,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using custom inputs
    enable_prefix_caching=False,
    #load_format="dummy",  # Use dummy loader for random weights
    trust_remote_code=True,
    #enforce_eager=True,
    compilation_config={"cudagraph_mode": "PIECEWISE"},
    input_coalesce_timeout_ms=5,
    shm_decode=True,
)

print("Initializing engine...")
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=max_len, skip_sampling=False)

print("Engine initialized successfully")


In [ ]:
# prepare prefill tokens
import json
from transformers import AutoTokenizer
from vllm.model_executor.models.qwen3_tts import Qwen3TTSTalkerForConditionalGeneration


with open(config_path / "config.json") as f:
  config = json.load(f)
tokenizer = AutoTokenizer.from_pretrained(str(config_path.absolute()))
text_ids, acoustic_ids = Qwen3TTSTalkerForConditionalGeneration.build_prefill_tokens(
  tokenizer=tokenizer,
  text="Another example, to see that everything is actually fine before moving this to the server as well.",
  speaker="aiden",
  language="english",
  config=config,
)
print(f"Prefill text_ids {text_ids.shape}, acoustic_ids {acoustic_ids.shape}")

In [ ]:
request_id = "test_request_1"

tc = config["talker_config"]
codec_eos_token_id = tc["codec_eos_token_id"]
tts_pad_token_id = config["tts_pad_token_id"]
_W = config.get("repetition_window", 256)
num_code_groups = tc.get("num_code_groups", 16)
zero_token = tc["acoustic_zero_token_id"]

prompt_len = text_ids.shape[0]

inputs = {
    "prompt_token_ids": [0] * prompt_len,
    "custom_inputs": {
        "text_ids": text_ids,
        "acoustic_ids": acoustic_ids,
        "prev_group0_tokens": torch.full(
            (prompt_len, _W), codec_eos_token_id + 1, dtype=torch.long),
    },
}

print(f"Starting generation with request_id: {request_id}")
print(f"Prompt length: {prompt_len}")

queue = await engine.add_request(request_id, inputs, sampling_params)
prefill_output = await queue.get()
custom_out = prefill_output.outputs[0].custom_outputs
first_token = custom_out["codes"][-1:]  # [1, 16]

generated_codecs = [first_token]

prev_g0 = torch.full((1, _W), codec_eos_token_id + 1, dtype=torch.long)
g0_tok = first_token[0, 0].item()
prev_g0[0, 0] = g0_tok
g0_write_pos = 1

decode_text_id = torch.tensor([tts_pad_token_id], dtype=torch.long)
decode_acoustic_id = first_token.clone()  # [1, 16] - all codec groups from previous step

for step in range(max_len - 1):
    outputs = engine.decode_step_shm(
        request_id,
        custom_inputs={
            "text_ids": decode_text_id,
            "acoustic_ids": decode_acoustic_id,
            "prev_group0_tokens": prev_g0,
        },
    )

    next_tokens = outputs["codes"][-1:].clone()  # [1, 16]
    g0_tok = next_tokens[0, 0].item()

    if g0_tok == codec_eos_token_id:
        print(f"EOS token detected at step {step + 1}, stopping generation.")
        await engine.abort(request_id)
        break

    generated_codecs.append(next_tokens)
    decode_acoustic_id = next_tokens  # feed all 16 groups back as input
    prev_g0[0, g0_write_pos % _W] = g0_tok
    g0_write_pos += 1
else:
    print(f"Max tokens ({max_len}) reached without EOS.")
    await engine.abort(request_id)

In [ ]:
arr = torch.cat(generated_codecs, dim=0)

import matplotlib.pyplot as plt

plt.imshow(arr.cpu().numpy().T, aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
print(arr.shape)
print(torch.min(arr), torch.max(arr))
torch.save(arr, "/home/vklimkov/workspace/qwen3_tts/Qwen3-TTS/vllm_pred_tokens.pt")